# Nova Data Pipeline

[Link 1](https://jwst-docs.stsci.edu/accessing-jwst-data#gsc.tab=0) <br>
[Link 2](https://outerspace.stsci.edu/spaces/MASTDOCS/pages/94962825/Portal+Guide)

In [1]:
pip install astroquery

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from astroquery.mast import Observations


def fetch_jwst_raw(instruments=("NIRCAM", "MIRI"), target=None, limit=10):
    print("Querying JWST observations...")

    # Query broad JWST observations first, then filter locally.
    obs = Observations.query_criteria(obs_collection="JWST")

    if len(obs) == 0:
        print("No JWST observations found")
        return

    instrument_tokens = tuple(i.upper() for i in instruments)
    obs = obs[[any(tok in str(inst).upper() for tok in instrument_tokens)
               for inst in obs["instrument_name"]]]

    if target:
        target_lower = target.lower()
        obs = obs[[target_lower in str(t).lower() for t in obs["target_name"]]]

    print(f"Observations after filters: {len(obs)}")
    if len(obs) == 0:
        print("No matching observations for selected instruments/target")
        return

    products = Observations.get_product_list(obs)

    # Unprocessed JWST exposures are usually *_uncal.fits products.
    uncal = Observations.filter_products(
        products,
        calib_level=[1],
        productSubGroupDescription="UNCAL",
        extension="fits"
    )

    print(f"UNCAL files found: {len(uncal)}")
    if len(uncal) == 0:
        print("No UNCAL files matched the criteria")
        return

    to_download = uncal[:limit]
    print(f"Downloading {len(to_download)} files...")
    manifest = Observations.download_products(to_download)
    return manifest

In [ ]:
# Example run
manifest = fetch_jwst_raw(
    instruments=("NIRCAM", "MIRI"),
    target="SMACS 0723",
    limit=5,
)
manifest

Querying JWST observations...
